# FedSwarm — Phase 3 FL smoke test (Colab, CPU-friendly)

Runs the Flower FL harness (`src/fedswarm/fl/app.py`) end-to-end: `FedAvg` over the
smoke-sized default in `pyproject.toml`'s `[tool.flwr.app.config]` (2 clients, 2
rounds, SimpleCNN@112). **Works on a CPU-only runtime with zero code changes** --
every device lookup in `fl/app.py` already falls back to CPU automatically, and
`ray` (Flower's simulation backend) needs no GPU either. Use this notebook instead of
`colab_centralized_training.ipynb` when GPU quota is tight; switch back for the
actual centralized/FL training sweeps once quota resets.

**Before running: select Runtime > Change runtime type > CPU.** Every GPU-runtime
attempt of this notebook failed with a "No heartbeat received from the task" error
(`docs/OPEN_QUESTIONS.md` has the investigation); the first genuinely successful run
was on a CPU-only runtime. This smoke config doesn't need a GPU anyway, so there's no
downside to picking CPU explicitly rather than leaving it on whatever the last
notebook set.

Have your dataset zip ready to upload (same one used to build the manifest locally),
unless it's already checkpointed to Drive from a previous session.

✅ Verified: `results/fl/e35285b33a_0.json` (gitignored, kept locally) is a real
completed 2-round run against this repo's own `phase-4` commit -- Phase 3's
acceptance test has passed at least once, on CPU.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/researchpaper784-alt/ResearchPaper.git"
REPO_DIR = "/content/ResearchPaper"

# Colab sessions can survive a cell re-run (e.g. retrying after an earlier cell
# failed) without wiping /content, so a plain `git clone` here fails with exit
# code 128 ("destination path already exists and is not an empty directory") on a
# rerun. Make this idempotent: pull if it's already a clone of this repo, re-clone
# if the directory exists but isn't (a partial/failed prior clone), else clone.
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # A force-push (e.g. a rebased history) makes --ff-only fail even on a
    # legitimate clone of this repo; re-clone rather than hard-failing the cell.
    pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    if pull.returncode != 0:
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
elif os.path.isdir(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)


In [ ]:
%cd /content/ResearchPaper

# flwr[simulation] pulls in `ray` plus its own transitive deps -- resolved normally
# (no --no-deps) since flwr itself pins nothing Colab-incompatible. Installed BEFORE
# fedswarm and as its own pip call for exactly that reason.
!pip install -q "flwr[simulation]>=1.36.0,<1.37.0"

# fedswarm's own [project.dependencies] pin torch==2.2.2 / numpy<2 for Intel-macOS
# wheel availability (the dev machine this repo was written on) -- neither version
# exists for Colab's Python/CUDA build at all (confirmed: `pip install -e ".[simulation]"`
# without --no-deps fails outright trying to resolve torch==2.2.2). Colab's base image
# already has a newer, working torch/numpy, and the line above just installed flwr --
# --no-deps skips fedswarm's pinned list entirely and reuses what is already present,
# exactly like colab_centralized_training.ipynb's install cell already does.
!pip install -q -e . --no-deps


## Stage checkpoints (run this first)

Colab's free tier recycles the runtime when the GPU quota runs out, wiping all of
`/content` without warning. Each expensive stage is therefore mirrored to Drive as
soon as it completes, and every cell below skips itself when its stage is already
satisfied:

| stage | cost if lost | restoring it skips |
|---|---|---|
| `dataset` | a 164MB upload | the upload cell |
| `cache` | JPEG decoding, and needs `dataset` | upload **and** extraction |

⚠️ This notebook's own FL results/checkpoints (`results/fl/`) are **not** mirrored to
Drive the way `results/centralized/` is -- a lost runtime mid-run costs the in-flight
FL run (cheap to redo: this is a 2-round smoke config). Wiring that up belongs to
Phase 6's real sweep runner, not this smoke test.


In [ ]:
import os
import sys

# Re-running this cell in a session where Drive is already mounted makes
# drive.mount() raise "Mountpoint must not already contain files", so check first.
if os.path.isdir("/content/drive/MyDrive"):
    print("Drive already mounted.")
else:
    from google.colab import drive

    drive.mount("/content/drive")

# Two import paths, for two different consumers. `pip install -e .` above makes
# fedswarm importable to the *child* interpreters every stage below shells out to.
# It does nothing for *this* kernel, which started before the install ran and so
# never scanned the new .pth file -- hence the src entry for in-kernel imports.
sys.path.insert(0, "src")

from fedswarm.utils.checkpoint import Checkpoint

ckpt = Checkpoint("/content/drive/MyDrive/fedswarm_backup")
print("restored:", ckpt.restore() or "nothing (first run)")
print()
print(ckpt.report())


## Upload the dataset

Run this cell either way -- it no-ops when the `dataset` stage was restored, and
otherwise opens a file picker. Select the Brain Tumor MRI zip (~164MB), the same
one used locally to build `manifest.csv`. It is saved as `data/raw/dataset.zip`
and checkpointed, so this upload is a one-time cost across all future sessions
(shared with `colab_centralized_training.ipynb` -- the same Drive backup covers both).


In [ ]:
from pathlib import Path

DATASET_ZIP = Path("data/raw/dataset.zip")

if DATASET_ZIP.exists():
    print(f"{DATASET_ZIP} already present -- skipping the upload.")
else:
    from google.colab import files

    uploaded = files.upload()
    DATASET_ZIP.parent.mkdir(parents=True, exist_ok=True)
    DATASET_ZIP.write_bytes(next(iter(uploaded.values())))
    print(f"saved {DATASET_ZIP} ({DATASET_ZIP.stat().st_size / 1e6:.1f} MB)")
    print("checkpointed:", ckpt.save("dataset"))


In [ ]:
import os
import subprocess
from pathlib import Path

os.environ["FEDSWARM_DATA_ROOT"] = "/content/brain-tumor-mri"

# Gated on the raw dataset root, not the cache -- a resumed session restores the
# cache from Drive but never the extracted raw images (correctly: they're derived
# from the zip, not their own checkpointed stage).
root = Path(os.environ["FEDSWARM_DATA_ROOT"])
if root.exists() and any(root.iterdir()):
    print(f"{root} already populated -- skipping extraction.")
else:
    proc = subprocess.run(
        ["python", "-m", "fedswarm.data.download",
         "--zip", str(DATASET_ZIP),
         "--root", str(root)],
        capture_output=True, text=True,
    )
    print(proc.stdout, proc.stderr)
    proc.check_returncode()

    # --zip implies --verify, which rewrites the tracked DATASET_CARD.md with this
    # host's scan and never gets committed -- revert it so provenance stays clean.
    subprocess.run(["git", "checkout", "--", "data/raw/DATASET_CARD.md"], check=False)


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "Running on CPU -- fine for this smoke config (2 clients, 2 rounds, "
        "SimpleCNN@112). Every device lookup in fl/app.py already falls back to "
        "CPU automatically; nothing here needs a GPU."
    )


## Build the decoded-image cache

Uses the manifest already committed to the repo (`data/processed/manifest.csv`).
Shared cache format with `colab_centralized_training.ipynb` -- if you've already
built/restored the 112px cache there, this cell no-ops here too.


In [ ]:
import subprocess

proc = subprocess.run(
    ["python", "-c", "from fedswarm.data.cache import ensure_cache; ensure_cache(112)"],
    capture_output=True, text=True,
)
print(proc.stdout, proc.stderr)
proc.check_returncode()

print("checkpointed:", ckpt.save("cache"))


## Run the FL smoke test

`flwr run .` reads `[tool.flwr.app]` in `pyproject.toml` -- 2 clients, 2 rounds,
`FedAvg`, SimpleCNN@112, IID partition. Override anything without editing files, e.g.:

```
!flwr run . --run-config "num-rounds=5 num-clients=4" --stream
```

Writes one result JSON to `results/fl/` on completion (same schema
`scripts/run_experiment.py`'s centralized runs use). `--stream` shows client/server
print output live instead of only at the end.


In [ ]:
import os

# flwr run executes an *installed copy* of this app (see fl/app.py's module
# docstring, 2026-09-16 note) from a location with no data/ or results/
# directory next to it at all -- every relative run_config path needs an
# explicit anchor back to this actual clone, or it silently resolves nowhere.
os.environ["FEDSWARM_REPO_ROOT"] = "/content/ResearchPaper"

# 2026-09-17: every GPU-runtime attempt failed with "No heartbeat received
# from the task" inside flwr run's per-run isolated environment
# (docs/OPEN_QUESTIONS.md has the full investigation). The first genuinely
# successful run (results/fl/e35285b33a_0.json, status "completed") was on a
# CPU-only runtime -- Runtime > Change runtime type > CPU, done BEFORE
# running this notebook's install cells. Use CPU runtime for this smoke test;
# it doesn't need a GPU anyway (2 clients, 2 rounds, SimpleCNN@112). Kept
# below regardless, since it's a real, harmless lever either way.
os.environ["FLWR_DISABLE_RUNTIME_DEPENDENCY_INSTALLATION"] = "1"

!flwr run . --stream

## Getting results back

Zips `results/fl/*.json` and triggers a browser download directly.


In [ ]:
from google.colab import files

!cd results && zip -r /content/fl_results.zip fl/ -x "fl/_checkpoints/*"
files.download("/content/fl_results.zip")


## The real-data validation run (do this before Phase 6)

Everything Phase 4 and 5 verified was on plumbing: FedACO has never trained on real
images. Two runs -- FedACO and FedAvg on the same seed and partition -- then a health
check that answers the three questions deciding whether Phase 6 is worth building:

1. **Is the colony searching?** `pheromone_entropy` pinned near its ceiling log(11)=2.398
   means tau is uniform, `tau^a * eta^b` collapses to `eta^b`, and there is no colony
   search or cross-round stigmergy -- the paper's actual contribution.
2. **Has the deposit floor engaged?** `best_fitness <= 0` zeroes every deposit.
3. **Does FedACO beat FedAvg, and is the fallback carrying it?** A high `fallback_used`
   rate means FedACO *was* FedAvg for most of the run.

⚠️ **Set client resources first.** The Simulation Runtime assigns **2 CPUs per
ClientApp** by default, so K=10 requests 20 cores -- far more than this runtime has, and
it does not degrade gracefully: it sits at round 0 with idle actors and no error. Pin it
to 1 and keep K within reach of `!nproc`. On a 2-core Colab runtime, start at `K = 4`.


In [ ]:
import os, subprocess

os.environ["FEDSWARM_REPO_ROOT"] = "/content/ResearchPaper"
os.environ["FLWR_DISABLE_RUNTIME_DEPENDENCY_INSTALLATION"] = "1"

print("cores available:", os.cpu_count())

# 2 CPUs/ClientApp is the default and is what makes larger K stall silently.
!flwr federation simulation-config --client-resources-num-cpus 1

K = 4          # keep this near `nproc`; raise only if rounds complete at a sane pace
ROUNDS = 15

for strategy in ("fedaco", "fedavg"):
    cfg = (
        f"strategy-name='{strategy}' num-clients={K} min-train-nodes={K} "
        f"min-evaluate-nodes={K} min-available-nodes={K} num-rounds={ROUNDS} "
        f"local-epochs=1 regime='dirichlet' alpha=0.3 seed=0"
    )
    print(f"\n=== {strategy} ===", flush=True)
    # --stream is required, not cosmetic: a bare `flwr run` returns exit 0 immediately
    # while the run is still starting, and the orphaned run keeps consuming the machine.
    subprocess.run(["flwr", "run", ".", "--stream", "--run-config", cfg], check=False)


In [ ]:
# Reads the result files and answers the three questions above.
# Exits nonzero with --strict if the colony is not searching.
!python scripts/check_fedaco_health.py --results-dir results/fl --out results/fedaco_health.json
